# Single-Run CLT vs Multi-Run Confidence Intervals

## The two estimators of the per-run standard error

Let $Z_i = e^{-r\Delta t}\,CF_1^i$ be the discounted eventual exercise cashflow for path $i$, so the LSM point estimate from one simulation of $N$ paths is

$$\bar{Z} = \frac{1}{N}\sum_{i=1}^N Z_i.$$

Let $\sigma_Z$ denote the (unknown) population standard deviation of $Z$. The **true** standard error of $\bar{Z}$ — the quantity we want to estimate — is

$$\mathrm{SE}(\bar{Z}) \;=\; \frac{\sigma_Z}{\sqrt{N}}.$$

We have two ways of estimating it:

**Single-run CLT estimate** (from the $N$ cashflows inside one simulation):

$$\widehat{\mathrm{SE}}_{\mathrm{single}} \;=\; \frac{s_Z}{\sqrt{N}}, \qquad s_Z = \sqrt{\tfrac{1}{N-1}\sum_i (Z_i-\bar Z)^2}.$$

This is what `amOptPricer.price_american_put_lsm_llh` returns in `std_err`.

**Multi-run empirical estimate** (from $R$ independent simulations, each with $N$ paths):

$$\widehat{\mathrm{SE}}_{\mathrm{multi}} \;=\; s_{\bar Z} \;=\; \sqrt{\tfrac{1}{R-1}\sum_{r=1}^R(\bar Z_r - \bar{\bar Z})^2}.$$

Here $\bar Z_1,\ldots,\bar Z_R$ are the $R$ single-run price estimates and $\bar{\bar Z}$ is their mean.

## Why these should agree

Both target the same quantity $\sigma_Z/\sqrt{N}$:

- $\widehat{\mathrm{SE}}_{\mathrm{single}} \xrightarrow{p} \sigma_Z/\sqrt{N}$ as $N\to\infty$ — within a single run, $s_Z$ converges to $\sigma_Z$.
- $\widehat{\mathrm{SE}}_{\mathrm{multi}} \xrightarrow{p} \sigma_Z/\sqrt{N}$ as $R\to\infty$ — across runs, the empirical SD of the $\bar Z_r$ converges to the true SD of $\bar Z$, which is $\sigma_Z/\sqrt{N}$.

If the two estimates agree in practice, the single-run CLT SE is a faithful measure of the price estimator's uncertainty. If they diverge, the single-run SE is missing something — most likely the variability injected by re-fitting the LSM exercise boundary on each new sample of paths (the $Z_i$ are **not** strictly i.i.d. because each depends on the regression fit).

## The ratio

$$\text{Ratio} \;=\; \frac{\widehat{\mathrm{SE}}_{\mathrm{multi}}}{\overline{\widehat{\mathrm{SE}}}_{\mathrm{single}}},
\qquad \overline{\widehat{\mathrm{SE}}}_{\mathrm{single}} \;=\; \frac{1}{R}\sum_r \widehat{\mathrm{SE}}_{\mathrm{single}}^{(r)}.$$

Under the CLT, ratio $\to 1$ as $N$ grows. A ratio noticeably above 1 flags that single-run CIs understate the true uncertainty (regression noise not captured).

## Axes of comparison

- **Regression basis:** Laguerre polynomials vs Gaussian RBF. Different bases have different in-sample regression noise profiles.
- **Moneyness:** ITM ($S_0 < K$) vs OTM ($S_0 > K$) puts. OTM has fewer ITM paths per step $\to$ fewer regression observations $\to$ potentially larger regression noise.
- **With vs without CV-LLH** (Section 3 stress test): the CV layer adds a second in-sample regression (the global $\hat\theta$), which could amplify the effect. Deep OTM ($S_0=115$, $K=100$) is the hardest case.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import testing as tst

PARAMS = dict(r=0.01, kappa=5, nu=0.2, lam=0.9, eta=0.01, rho=-0.2,
              sigma0=0.15, theta0=0.18)
K, T, N_STEPS_MC = 100.0, 1.0, 52
LLH_PARAMS = dict(phi_max=300.0, n_phi=513, n_steps_rk4=128)
R = 20

## 1. Main sweep: Plain LSM × {Laguerre, Gaussian} × {ITM, OTM}

For each basis and each moneyness, run $R=20$ independent simulations at $N \in \{1k, 5k, 10k, 50k\}$. The resulting table gives both SE measures side by side.

In [ ]:
S0_cases = [(90.0, 'ITM'), (110.0, 'OTM')]   # puts: S0<K ITM, S0>K OTM
N_values = [1_000, 5_000, 10_000, 50_000]

CI_CONFIGS = [
    ('Laguerre Plain', 'laguerre',  3, 1e-5, False),
    ('Gaussian Plain', 'gaussian', 15, 1e-4, False),
]

ci_df = tst.ci_comparison_grid(PARAMS, K, T, N_STEPS_MC,
                               S0_cases, N_values, R, CI_CONFIGS)
tst.format_ci_table(ci_df)

## 2. SE levels — multi-run vs single-run CLT

**Red solid** = multi-run SE (true variability, including regression error).
**Blue dashed** = mean of single-run CLT SEs (ignores regression error).

If the two curves coincide, the CLT approximation is adequate. A persistent gap signals that single-run CIs understate the true uncertainty.

In [ ]:
tst.plot_ci_levels(ci_df, S0_cases, R)

## 3. Deep OTM stress test: all 4 basis × CV combinations

Deep OTM ($S_0 = 115$, $K = 100$) has very few ITM paths per step, which stresses the regression. If the CLT approximation survives here, it survives everywhere. We run all four combinations at $N = 10{,}000$ with $R = 20$ replications.

In [ ]:
import numpy as np

STRESS_CONFIGS = [
    ('Laguerre Plain',   'laguerre',  3, 1e-5, False),
    ('Laguerre CV-LLH',  'laguerre',  3, 1e-5, True),
    ('Gaussian Plain',   'gaussian', 15, 1e-4, False),
    ('Gaussian CV-LLH',  'gaussian', 15, 1e-4, True),
]

stress_rows = []
for name, btype, border, ridge, use_cv in STRESS_CONFIGS:
    prices, ses = tst.run_multi_basis(
        PARAMS, 115.0, K, T, N_STEPS_MC, 10_000, R,
        basis_type=btype, basis_order=border, ridge=ridge,
        use_cv=use_cv, llh_params=LLH_PARAMS)
    se_multi = prices.std(ddof=1) / np.sqrt(R)
    se_single = ses.mean()
    stress_rows.append({
        'Config': name,
        'Mean price': round(prices.mean(), 4),
        'SE_multi': round(se_multi, 4),
        'SE_single (mean)': round(se_single, 4),
        'Ratio': round(se_multi / se_single, 3) if se_single > 0 else np.nan,
    })
    print(f"  {name}: mean={prices.mean():.4f}, ratio={se_multi/se_single:.3f}")

pd.DataFrame(stress_rows).set_index('Config')